# Day 2: Back-and-Forth Chat Practice (Guided)

Goal for today: learn how context is managed across turns in a chatbot without copying a full solution.

Rules for this notebook:
- You complete the TODOs.
- Use clues to reason about each step.
- Keep each testable step small so debugging is easier.
- Use `uv` for environment and dependency management.

## 1) Setup

You already know `chat.completions.create`, so focus on how messages are built over time.

Clue:
- If your notebook is in `tests/`, your `.env` path is usually one level up.
- Run dependencies with uv (from repo root): `uv add --project learn-langfuse gradio`
- Run Jupyter with uv project env: `uv run --project learn-langfuse jupyter lab`

In [1]:
from dotenv import load_dotenv
from langfuse.openai import OpenAI
from langfuse import get_client
import gradio as gr
import os

# TODO: load your environment variables from the repo root .env
# Hint: load_dotenv("../.env")
load_dotenv("../.env")

# TODO: initialize your OpenAI client (Langfuse-wrapped OpenAI)
openai = OpenAI()

# TODO: initialize Langfuse client
langFuse = get_client()

# Clue: Langfuse needs LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST in .env

# Quick self-check (do not print your key):
hasOpenAiKey = bool(os.getenv("OPENAI_API_KEY"))
hasLangfusePublicKey = bool(os.getenv("LANGFUSE_PUBLIC_KEY"))
hasLangfuseSecretKey = bool(os.getenv("LANGFUSE_SECRET_KEY"))

## 2) Baseline: Stateless Bot

Start with a bot that answers only the current message.

Why this matters:
- This gives you a baseline behavior to compare against once you add context.
- If something breaks later, return here first.

In [5]:
def StatelessReply(userMessage):
    # TODO: call chat.completions.create using ONLY the latest user message
    userPrompt = {
        "role": "user",
        "content": userMessage
    }
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[userPrompt]
    )

    # TODO: return assistant text
    assistantText = response.choices[0].message.content

    # TODO: flush Langfuse client so traces show up immediately while learning
    langFuse.flush()
    return assistantText

# TODO: uncomment after implementing
demoStateless = gr.Interface(fn=StatelessReply, inputs="text", outputs="text")
demoStateless.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 3) Move to Conversational State

Now switch to `gr.ChatInterface`, which gives you `message` and `history`.

Core idea:
- `history` is what lets the model remember prior turns (if you pass it to the API call).
- If you ignore `history`, your bot acts stateless even in a chat UI.

In [21]:
def BuildMessagesWithContext(message, history):
    """
    TODO exercise:
    1) Start with a system message (persona/rules).
    2) Convert each (user, assistant) pair in history into API message objects.
    3) Append the latest user message at the end.

    Output format target:
    [
      {"role": "system", "content": "..."},
      {"role": "user", "content": "..."},
      {"role": "assistant", "content": "..."},
      ...
    ]
    """

    # define the base history array to hold each user prompt and assitant response as message objects
    messageHistory = []         

    # we iterate over every user pompt, and each assistant response in the history
    for request in history:

        role = request.get("role")
        content = request.get("content")
        if role and content:
            messageHistory.append({
                "role": role,
                "content": content
            })

    # print the message history to see the format before sending to the API
    print(messageHistory)
    
    #this is then added to messageHistory, which is then sent to the API as the messages parameter, along with the system prompt.

    userPrompt = {
        "role": "user",
        "content": message
    }

    systemPrompt = {
        "role": "system",
        "content": "You are an excellent joke writer. You write dark jokes for dark humor lovers. You never break character."
    }

    messageHistory.insert(0, systemPrompt) # we insert the system prompt at the beginning of the message history, so that it is the first thing the model sees, and sets the tone for the conversation.
    messageHistory.append(userPrompt) # we append the latest user prompt at the end of the message history, so that it is the last thing the model sees, and is the most relevant to generating the next response.

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messageHistory
    )

    # always flush after your API call, so that you can see the trace in Langfuse immediately while learning!
    langFuse.flush()

    # never forget to return the model's response!
    chatResponse = response.choices[0].message.content
    return chatResponse

demoHistory = gr.ChatInterface(fn=BuildMessagesWithContext)
demoHistory.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## 4) Control Context Window (Most Important Part)

You now need control, not just memory.

Problem:
- Sending full history forever increases token usage and can hurt reliability.

Exercise:
- Keep only the last `N` exchanges from history before building messages.
- Start with `N = 4` and compare behavior/cost with larger N.

In [ ]:
def TrimHistory(history, maxExchanges=4):
    # TODO: return only the newest maxExchanges user-assistant pairs
    # Hint: slicing is enough for this exercise
    keep = maxExchanges * 2
    return history[-keep:]

def ChatWithTrimmedContext(message, history):
    # TODO: trim history first, then build messages, then call API
    # TODO: flush Langfuse client
    trimmedHistory = TrimHistory(history)
    response = BuildMessagesWithContext(message, trimmedHistory)
    return response

demoTrimmedHistory = gr.ChatInterface(fn=ChatWithTrimmedContext)
demoTrimmedHistory.launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'content': [{'text': 'my secret code is ZX-941-KITE.\n\nTell me a joke!', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': "Why don’t graveyards have Wi-Fi? \n\nBecause people are just dying to get in, but they don't want to connect!", 'type': 'text'}]}]
[{'role': 'user', 'content': [{'text': 'my secret code is ZX-941-KITE.\n\nTell me a joke!', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': "Why don’t graveyards have Wi-Fi? \n\nBecause people are just dying to get in, but they don't want to connect!", 'type': 'text'}]}, {'role': 'user', 'content': [{'text': 'another one', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': 'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field... right until the crows started digging up his dark secrets!', 'type': 'text'}]}]
[{'role': 'user', 'content': [{'text': 'my secret code is ZX-941-KITE.\n\nTell me a joke!', 'type': 'text'}]}, {'role': 'assistant', 'content': [{

## 5) Debugging Checklist (For Real Troubleshooting)

When bugs appear, inspect in this order:
1. Is `history` arriving as expected? (print first, then remove prints)
2. Are your roles ordered correctly: system -> past turns -> latest user?
3. Did you accidentally duplicate the latest user message?
4. Are you trimming too aggressively (losing key context)?
5. Are model responses empty due to API/model/permission errors?
6. Is trace visible in Langfuse dashboard after each turn (flush + valid keys)?

In [19]:
def InspectPayloadShape(sampleMessage, sampleHistory):
    # TODO: build payload using your function and print it clearly
    # Tip: verify role order manually before calling the model
    raise NotImplementedError("Implement InspectPayloadShape")

## 6) Reflection Prompts

Answer these after finishing:
- What changed in behavior from stateless to stateful?
- Which bug was hardest: payload shape, role ordering, or trimming? Why?
- What context policy would you use in a production app and why?